In [1]:
from kaggle_secrets import UserSecretsClient
import subprocess

pat = UserSecretsClient().get_secret("GITHUB_PAT")
repo = f"https://{pat}@github.com/msheibani111/thesis-stage1.git"

subprocess.run(["git", "clone", "--depth", "1", repo, "/kaggle/working/repo"],
               check=True, capture_output=True)

import sys
sys.path.insert(0, "/kaggle/working/repo")

from losses import get_quantiles, masked_pinball_loss
from heads import MonotoneQuantileHead
print("imported ok")

imported ok


In [2]:
!cd /kaggle/working/repo && python test_losses.py

quantile levels: (0.05, 0.25, 0.5, 0.75, 0.95)
test shapes: B=32, K=6, d=27, Q=5

pinball core
  pass  pinball asymmetry
  pass  perfect prediction gives exactly zero
  pass  loss is non-negative

masking
  pass  fill values cannot leak into the loss
  pass  fully masked batch does not produce NaN

empty channels
  pass  empty channel produces no NaN in forward or backward
  pass  empty channel is skipped, not scored as zero

aggregation
  pass  aggregations differ (per-channel 0.6315 > per-observation 0.5589)
  pass  per-channel breakdown shapes and finiteness

input validation
  pass  malformed inputs raise ValueError

heads
  pass  monotone head never crosses, across input scales
  pass  shape (32, 6, 27, 5), no parameter cost (52,650 either way)
  pass  trains end to end (0.5304 -> 0.0059), still no crossing
  pass  unconstrained head crosses (50.0% of pairs), as expected

reproducibility
  pass  deterministic under fixed seed

15 tests passed


In [3]:
import numpy as np, torch, json
from pathlib import Path
from losses import get_quantiles, masked_pinball_loss, per_channel_pinball

DATA = Path('/kaggle/input/datasets/mohammadsheibani/challenge-2019-stage1-tensors/tensors')
meta = json.load(open(DATA / 'tensor_meta.json'))
names = meta['target_channels']

tr = np.load(DATA / 'train.npz', allow_pickle=True)
te = np.load(DATA / 'test.npz',  allow_pickle=True)

q  = get_quantiles()
qn = q.numpy()
D  = len(names)

# fit marginal quantiles per channel on TRAIN only
const_q = np.zeros((D, len(qn)), dtype=np.float32)
Ytr, Mtr = tr['Y'], tr['Y_mask']
for j in range(D):
    obs = Ytr[:, :, j][Mtr[:, :, j] == 1]
    if obs.size:
        const_q[j] = np.quantile(obs, qn)

# evaluate on TEST
Yte = torch.from_numpy(te['Y'])
Mte = torch.from_numpy(te['Y_mask'])
N   = Yte.shape[0]
pred = torch.from_numpy(const_q).view(1, 1, D, len(qn)).expand(N, 6, D, len(qn)).contiguous()

pc = masked_pinball_loss(Yte, pred, Mte, q, per_channel=True)
po = masked_pinball_loss(Yte, pred, Mte, q, per_channel=False)
ch, cnt = per_channel_pinball(Yte, pred, Mte, q)

print(f"CONSTANT-QUANTILE BASELINE (test set, {N:,} windows)")
print(f"  per-channel     {pc:.4f}")
print(f"  per-observation {po:.4f}")
print(f"  ratio           {pc/po:.3f}x\n")

print(f"{'channel':<16}{'loss':>9}{'n_obs':>12}")
print("-" * 37)
for i in torch.argsort(cnt, descending=True):
    print(f"{names[i]:<16}{ch[i]:>9.4f}{int(cnt[i]):>12,d}")

assert bool((pred[..., 1:] - pred[..., :-1] >= 0).all()), "fitted quantiles cross"
print("\nfitted constants non-crossing: True")

CONSTANT-QUANTILE BASELINE (test set, 52,185 windows)
  per-channel     0.2343
  per-observation 0.2393
  ratio           0.979x

channel              loss       n_obs
-------------------------------------
HR                 0.2402     291,757
Resp               0.2406     285,767
MAP                0.2417     283,244
O2Sat              0.2375     277,112
SBP                0.2520     264,270
DBP                0.2480     174,005
Temp               0.2225     100,055
FiO2               0.1929      42,727
Hct                0.2266      30,740
Glucose            0.2324      30,390
pH                 0.2295      29,418
Potassium          0.2171      29,378
BaseExcess         0.2356      25,946
Hgb                0.2297      23,621
Magnesium          0.2011      22,895
Chloride           0.2413      22,658
BUN                0.2634      22,215
HCO3               0.2526      21,827
PaCO2              0.2499      21,654
WBC                0.2342      20,293
Creatinine         0.2437      18,